In [1]:
from pathlib import Path
import importlib.util

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from matplotlib.patches import Rectangle, FancyBboxPatch

# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "mp4"   # "gif" or "mp4"
THEME = "dark"

OUT_DIR = Path("media-site/animations/zeeman")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"zeeman_splitting.{OUTPUT_FORMAT}"

FPS = 24
FRAMES = 180
DPI = 160

LAMBDA_0 = 6562.8      # H-alpha, Angstrom
MAX_SPLIT = 0.55       # Angstrom, visualized splitting
SIGMA = 0.055          # line width

# =========================================================
# STYLE TOKENS
# =========================================================

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "style" / "hud_style_tokens.py").exists():
            return p
    return Path.cwd()


def load_tokens():
    path = find_project_root() / "style" / "hud_style_tokens.py"
    if not path.exists():
        return None

    spec = importlib.util.spec_from_file_location("hud_style_tokens", path)
    tokens = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tokens)
    print(f"Loaded style tokens from: {path}")
    return tokens


def mpl_color(value):
    if isinstance(value, tuple) and len(value) == 3:
        return tuple(v / 255 for v in value)
    return value


tokens = load_tokens()

if THEME == "dark":
    BG = mpl_color(getattr(tokens, "SA_COLORS_BG", (2, 7, 13))) if tokens else "#02070d"
    PANEL = "#00080e"
    TEXT = mpl_color(getattr(tokens, "SA_COLORS_TEXT_MAIN", (216, 251, 255))) if tokens else "#d8fbff"
    TEXT_DIM = mpl_color(getattr(tokens, "SA_COLORS_TEXT_DIM", (159, 199, 212))) if tokens else "#9fc7d4"
    CYAN = mpl_color(getattr(tokens, "SA_CHART_LINE_COLOR", (90, 240, 255))) if tokens else "#5af0ff"
    GRID = mpl_color(getattr(tokens, "SA_CHART_GRID_COLOR", (42, 215, 255))) if tokens else "#2ad7ff"
else:
    BG = "#f7fbff"
    PANEL = "#ffffff"
    TEXT = "#071a24"
    TEXT_DIM = "#33515c"
    CYAN = "#007a9a"
    GRID = "#8aa9b3"

BLUE = "#1e78ff"
GREEN = "#52ff52"
RED = "#ff3040"
YELLOW = "#ffd85a"

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "DejaVu Sans Mono",
    "mathtext.fontset": "dejavusans",
    "figure.facecolor": BG,
    "axes.facecolor": PANEL,
    "savefig.facecolor": BG,
    "text.color": TEXT,
    "axes.labelcolor": TEXT_DIM,
    "xtick.color": TEXT_DIM,
    "ytick.color": TEXT_DIM,
    "axes.edgecolor": GRID,
})

# =========================================================
# MODEL
# =========================================================

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def gaussian(x, mu, sigma, amp=1.0):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def zeeman_spectrum(wave, split):
    """
    Normal Zeeman triplet:
    sigma-, pi, sigma+.
    For split=0 components merge into one line.
    """
    if split < 1e-4:
        return gaussian(wave, LAMBDA_0, SIGMA, 1.0)

    y = (
        gaussian(wave, LAMBDA_0 - split, SIGMA, 0.72) +
        gaussian(wave, LAMBDA_0,         SIGMA, 0.95) +
        gaussian(wave, LAMBDA_0 + split, SIGMA, 0.72)
    )
    return y / y.max()


def magnetic_field_profile(frame):
    p = frame / max(1, FRAMES - 1)

    if p < 0.20:
        return 0.0

    if p < 0.72:
        return smoothstep((p - 0.20) / 0.52)

    return 0.92 + 0.08 * np.sin(2 * np.pi * (p - 0.72) / 0.28)


# =========================================================
# FIGURE
# =========================================================

fig = plt.figure(figsize=(16, 9), facecolor=BG)

ax_main = fig.add_axes([0.055, 0.13, 0.63, 0.56])
ax_levels = fig.add_axes([0.055, 0.735, 0.34, 0.17])
ax_telemetry = fig.add_axes([0.72, 0.13, 0.225, 0.76])
ax_pol = fig.add_axes([0.42, 0.735, 0.265, 0.17])

for ax in [ax_main, ax_levels, ax_telemetry, ax_pol]:
    ax.set_facecolor(PANEL)
    for spine in ax.spines.values():
        spine.set_color(CYAN)
        spine.set_alpha(0.45)
    ax.tick_params(colors=TEXT_DIM)

fig.text(
    0.055, 0.955,
    "ZEEMAN SPLITTING // SPECTRAL LINE RESPONSE",
    color=CYAN,
    fontsize=22,
    weight="bold",
    ha="left",
    va="top"
)

fig.text(
    0.055, 0.918,
    "synthetic telemetry animation · magnetic field induced spectral triplet",
    color=TEXT_DIM,
    fontsize=10,
    ha="left",
    va="top"
)

# =========================================================
# MAIN SPECTRUM PANEL
# =========================================================

wave = np.linspace(6561.7, 6563.9, 1600)

ax_main.set_xlim(wave.min(), wave.max())
ax_main.set_ylim(-0.12, 1.28)
ax_main.set_xlabel("Wavelength λ  [Å]")
ax_main.set_ylabel("Normalized intensity")
ax_main.set_title("OBSERVED SPECTRUM: Hα LINE", color=CYAN, loc="left", pad=12)
ax_main.grid(True, color=GRID, alpha=0.12, lw=0.7)

spectrum_line, = ax_main.plot([], [], color=YELLOW, lw=2.2)
sigma_minus_line, = ax_main.plot([], [], color=BLUE, lw=1.5, alpha=0.0)
pi_line, = ax_main.plot([], [], color=GREEN, lw=1.5, alpha=0.0)
sigma_plus_line, = ax_main.plot([], [], color=RED, lw=1.5, alpha=0.0)

scan_line = ax_main.axvline(wave.min(), color=CYAN, lw=1.0, alpha=0.35)

label_sigma_minus = ax_main.text(0, 1.08, r"$\sigma^-$", color=BLUE, ha="center", fontsize=13, alpha=0)
label_pi = ax_main.text(0, 1.14, r"$\pi$", color=GREEN, ha="center", fontsize=13, alpha=0)
label_sigma_plus = ax_main.text(0, 1.08, r"$\sigma^+$", color=RED, ha="center", fontsize=13, alpha=0)

ax_main.text(
    0.02, 0.04,
    r"$\Delta\lambda \propto g_{\rm eff}\,m\,B$",
    transform=ax_main.transAxes,
    color=TEXT_DIM,
    fontsize=12,
    alpha=0.75
)

# =========================================================
# ENERGY LEVELS PANEL
# =========================================================

ax_levels.set_xlim(0, 1)
ax_levels.set_ylim(0, 1)
ax_levels.axis("off")
ax_levels.set_title("ENERGY LEVELS", color=CYAN, loc="left", pad=8)

level_base_1, = ax_levels.plot([0.08, 0.32], [0.30, 0.30], color=TEXT_DIM, lw=1.2)
level_base_2, = ax_levels.plot([0.08, 0.32], [0.72, 0.72], color=TEXT_DIM, lw=1.2)

split_lines = []
for y0 in [0.30, 0.72]:
    for col in [BLUE, GREEN, RED]:
        ln, = ax_levels.plot([], [], color=col, lw=1.4, alpha=0)
        split_lines.append(ln)

transition_lines = []
for col in [BLUE, GREEN, RED]:
    ln, = ax_levels.plot([], [], color=col, lw=1.0, alpha=0)
    transition_lines.append(ln)

ax_levels.text(0.08, 0.12, "B = 0", color=TEXT_DIM, fontsize=10)
field_text = ax_levels.text(0.62, 0.12, "B > 0", color=CYAN, fontsize=10, alpha=0)

# =========================================================
# POLARIZATION PANEL
# =========================================================

ax_pol.set_xlim(0, 1)
ax_pol.set_ylim(0, 1)
ax_pol.axis("off")
ax_pol.set_title("POLARIZATION COMPONENTS", color=CYAN, loc="left", pad=8)

pol_items = [
    (0.20, BLUE, r"$\sigma^-$", "circular L"),
    (0.50, GREEN, r"$\pi$", "linear"),
    (0.80, RED, r"$\sigma^+$", "circular R"),
]

pol_circles = []
for x0, col, title, subtitle in pol_items:
    circ = plt.Circle((x0, 0.52), 0.105, fill=False, color=col, lw=1.3, alpha=0.2)
    ax_pol.add_patch(circ)
    t1 = ax_pol.text(x0, 0.27, title, color=col, ha="center", fontsize=13, alpha=0.2)
    t2 = ax_pol.text(x0, 0.13, subtitle, color=TEXT_DIM, ha="center", fontsize=8, alpha=0.2)
    pol_circles.append((circ, t1, t2))

pol_arrows = []
for x0, col, _, _ in pol_items:
    q = ax_pol.quiver(
        [x0], [0.52], [0.001], [0.001],
        color=col,
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.006,
        alpha=0.0
    )
    pol_arrows.append(q)

# =========================================================
# TELEMETRY PANEL
# =========================================================

ax_telemetry.set_xlim(0, 1)
ax_telemetry.set_ylim(0, 1)
ax_telemetry.axis("off")
ax_telemetry.set_title("MAGNETIC FIELD TELEMETRY", color=CYAN, loc="left", pad=12)

def hud_box(ax, xy, wh, label):
    x, y = xy
    w, h = wh
    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.012,rounding_size=0.015",
        edgecolor=CYAN,
        facecolor=(0, 0, 0, 0),
        lw=0.8,
        alpha=0.45
    )
    ax.add_patch(rect)
    ax.text(x + 0.025, y + h - 0.07, label, color=TEXT_DIM, fontsize=9, ha="left")
    return rect

hud_box(ax_telemetry, (0.06, 0.76), (0.88, 0.15), "FIELD STRENGTH")
hud_box(ax_telemetry, (0.06, 0.56), (0.88, 0.15), "LINE SPLITTING")
hud_box(ax_telemetry, (0.06, 0.36), (0.88, 0.15), "STATE")
hud_box(ax_telemetry, (0.06, 0.10), (0.88, 0.20), "COMPONENTS")

b_value_text = ax_telemetry.text(0.10, 0.80, "", color=CYAN, fontsize=20, weight="bold")
dl_value_text = ax_telemetry.text(0.10, 0.60, "", color=YELLOW, fontsize=20, weight="bold")
state_text = ax_telemetry.text(0.10, 0.40, "", color=TEXT, fontsize=13)

component_texts = [
    ax_telemetry.text(0.10, 0.23, r"σ⁻  blue-shifted", color=BLUE, fontsize=10, alpha=0),
    ax_telemetry.text(0.10, 0.18, r"π   unshifted", color=GREEN, fontsize=10, alpha=0),
    ax_telemetry.text(0.10, 0.13, r"σ⁺  red-shifted", color=RED, fontsize=10, alpha=0),
]

b_bar_bg = Rectangle((0.10, 0.775), 0.72, 0.012, facecolor=(1, 1, 1, 0.08), edgecolor=CYAN, lw=0.5)
b_bar_fg = Rectangle((0.10, 0.775), 0.0, 0.012, facecolor=CYAN, edgecolor="none", alpha=0.75)
ax_telemetry.add_patch(b_bar_bg)
ax_telemetry.add_patch(b_bar_fg)

dl_bar_bg = Rectangle((0.10, 0.575), 0.72, 0.012, facecolor=(1, 1, 1, 0.08), edgecolor=YELLOW, lw=0.5)
dl_bar_fg = Rectangle((0.10, 0.575), 0.0, 0.012, facecolor=YELLOW, edgecolor="none", alpha=0.8)
ax_telemetry.add_patch(dl_bar_bg)
ax_telemetry.add_patch(dl_bar_fg)

# =========================================================
# ANIMATION
# =========================================================

def save_animation(anim, out_file: Path):
    if OUTPUT_FORMAT.lower() == "gif":
        writer = PillowWriter(fps=FPS)
        anim.save(out_file, writer=writer, dpi=DPI)
    elif OUTPUT_FORMAT.lower() == "mp4":
        writer = FFMpegWriter(
            fps=FPS,
            metadata=dict(artist="Stellar Attractor"),
            bitrate=2600,
        )
        anim.save(out_file, writer=writer, dpi=DPI)
    else:
        raise ValueError("OUTPUT_FORMAT must be 'gif' or 'mp4'")
    print(f"Saved: {out_file}")


def init():
    spectrum_line.set_data([], [])
    sigma_minus_line.set_data([], [])
    pi_line.set_data([], [])
    sigma_plus_line.set_data([], [])
    return []


def update(frame):
    p = frame / max(1, FRAMES - 1)
    b = magnetic_field_profile(frame)
    split = MAX_SPLIT * b

    y = zeeman_spectrum(wave, split)

    # subtle telemetry noise, not snake-like motion
    rng = np.random.default_rng(frame)
    noise = rng.normal(0, 0.0035, size=len(y))
    y_live = np.clip(y + noise * (0.2 + 0.8 * b), 0, None)

    spectrum_line.set_data(wave, y_live)

    if b < 0.03:
        spectrum_line.set_color(YELLOW)
    else:
        spectrum_line.set_color(CYAN)

    # component guide curves
    alpha_comp = smoothstep((b - 0.10) / 0.35)

    sigma_minus_line.set_data(wave, gaussian(wave, LAMBDA_0 - split, SIGMA, 0.72))
    pi_line.set_data(wave, gaussian(wave, LAMBDA_0, SIGMA, 0.95))
    sigma_plus_line.set_data(wave, gaussian(wave, LAMBDA_0 + split, SIGMA, 0.72))

    sigma_minus_line.set_alpha(0.65 * alpha_comp)
    pi_line.set_alpha(0.65 * alpha_comp)
    sigma_plus_line.set_alpha(0.65 * alpha_comp)

    # scanline
    scan_x = wave.min() + (wave.max() - wave.min()) * ((p * 2.0) % 1.0)
    scan_line.set_xdata([scan_x, scan_x])

    # labels
    label_alpha = smoothstep((b - 0.35) / 0.35)
    label_sigma_minus.set_position((LAMBDA_0 - split, 1.08))
    label_pi.set_position((LAMBDA_0, 1.14))
    label_sigma_plus.set_position((LAMBDA_0 + split, 1.08))

    label_sigma_minus.set_alpha(label_alpha)
    label_pi.set_alpha(label_alpha)
    label_sigma_plus.set_alpha(label_alpha)

    # energy levels
    for ln in split_lines:
        ln.set_alpha(0.85 * alpha_comp)

    offsets = [-0.10 * b, 0.0, 0.10 * b]
    colors = [BLUE, GREEN, RED]

    k = 0
    for base_y in [0.30, 0.72]:
        for off, col in zip(offsets, colors):
            split_lines[k].set_data([0.55, 0.82], [base_y + off, base_y + off])
            k += 1

    for ln, off, col in zip(transition_lines, offsets, colors):
        ln.set_alpha(0.75 * alpha_comp)
        ln.set_data([0.685, 0.685], [0.30 + off, 0.72 + off])

    field_text.set_alpha(alpha_comp)

    # polarization
    for idx, (circ, t1, t2) in enumerate(pol_circles):
        a = 0.25 + 0.75 * label_alpha
        circ.set_alpha(a)
        t1.set_alpha(a)
        t2.set_alpha(0.45 + 0.55 * label_alpha)

        phase = 2 * np.pi * (p * 2 + idx / 3)
        if idx == 1:
            u, v = 0.0, 0.13 * np.sin(phase)
        else:
            u, v = 0.11 * np.cos(phase), 0.11 * np.sin(phase)

        pol_arrows[idx].set_UVC([u], [v])
        pol_arrows[idx].set_alpha(label_alpha)

    # telemetry
    b_value_text.set_text(f"{100 * b:05.1f} %")
    dl_value_text.set_text(f"±{split:0.3f} Å")

    if b < 0.05:
        state = "B = 0 // SINGLE LINE"
    elif b < 0.65:
        state = "FIELD RISING // SPLIT ACTIVE"
    else:
        state = "FULL ZEEMAN TRIPLET"

    state_text.set_text(state)

    b_bar_fg.set_width(0.72 * b)
    dl_bar_fg.set_width(0.72 * b)

    for t in component_texts:
        t.set_alpha(label_alpha)

    return []


anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    init_func=init,
    interval=1000 / FPS,
    blit=False,
)

save_animation(anim, OUT_FILE)
plt.close(fig)

Loaded style tokens from: /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/style/hud_style_tokens.py
Saved: animations/zeeman/zeeman_splitting.mp4
